# 🛒 Exploratory Data Analysis — Online Retail II

This notebook walks through the UCI Online Retail II dataset to understand transaction patterns,
customer behaviour, and data quality issues before building the churn prediction model.

**Dataset**: [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) — ~1.07M transaction lines from a UK-based online retailer (Dec 2009 – Dec 2011)  
**Citation**: Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. CC BY 4.0.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

%matplotlib inline

## 1. Load Raw Data

In [ ]:
# Load both sheets of the Excel file
sheets = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

print(f'Total rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')
df.head()

In [ ]:
# Data types and missing values
print('Data types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nMissing customer IDs: {df["Customer ID"].isnull().sum():,} ({df["Customer ID"].isnull().mean():.1%})')

## 2. Data Quality Assessment

In [ ]:
# Check for duplicates (the two sheets overlap in Dec 2010)
n_dups = df.duplicated().sum()
print(f'Exact duplicate rows: {n_dups:,}')

# Check non-product stock codes
non_product = df[~df['StockCode'].astype(str).str.match(r'^\d{5}')]
print(f'Non-product stock codes: {len(non_product):,}')
print(non_product['StockCode'].value_counts().head(10))

# Zero/negative prices
print(f'\nRows with price <= 0: {(df["Price"] <= 0).sum():,}')
print(f'Rows with negative quantity (non-return): {((df["Quantity"] < 0) & ~df["Invoice"].astype(str).str.startswith("C")).sum():,}')

In [ ]:
# Clean for the rest of the EDA
clean = df.drop_duplicates()
clean = clean[clean['Customer ID'].notna()].copy()
clean['Customer ID'] = clean['Customer ID'].astype(int)
clean = clean[clean['StockCode'].astype(str).str.match(r'^\d{5}')]
clean = clean[clean['Price'] > 0]
clean['is_return'] = clean['Invoice'].astype(str).str.startswith('C').astype(int)
clean['LineAmount'] = clean['Quantity'] * clean['Price']
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'])
clean['YearMonth'] = clean['InvoiceDate'].dt.to_period('M')

purchases = clean[clean['is_return'] == 0].copy()
print(f'Clean purchase rows: {len(purchases):,}')
print(f'Unique customers: {purchases["Customer ID"].nunique():,}')
print(f'Unique invoices: {purchases["Invoice"].nunique():,}')

## 3. Transaction Volume Over Time

In [ ]:
monthly = purchases.groupby('YearMonth').agg(
    transactions=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique'),
    revenue=('LineAmount', 'sum')
).reset_index()
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, title, color in zip(axes,
    ['transactions', 'customers', 'revenue'],
    ['Monthly Transactions', 'Monthly Active Customers', 'Monthly Revenue (£)'],
    ['#2f6db5', '#d4692a', '#2ca02c']):
    ax.plot(monthly['YearMonth'], monthly[col], marker='o', ms=4, color=color)
    ax.set_title(title, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    if col == 'revenue':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
fig.tight_layout()
plt.show()

## 4. Customer Purchase Frequency Distribution

In [ ]:
cust_orders = purchases.groupby('Customer ID')['Invoice'].nunique().reset_index()
cust_orders.columns = ['Customer ID', 'n_orders']

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Distribution (capped at 50 for visibility)
axes[0].hist(cust_orders['n_orders'].clip(upper=50), bins=50, color='#2f6db5', edgecolor='white')
axes[0].set(xlabel='Number of orders', ylabel='Customers', title='Purchase Frequency Distribution')
axes[0].axvline(cust_orders['n_orders'].median(), color='red', ls='--', label=f'Median: {cust_orders["n_orders"].median():.0f}')
axes[0].legend()

# Cumulative
sorted_orders = cust_orders['n_orders'].sort_values()
axes[1].plot(sorted_orders.values, np.linspace(0, 1, len(sorted_orders)), color='#2f6db5')
axes[1].set(xlabel='Number of orders', ylabel='Cumulative fraction of customers',
            title='CDF of Purchase Frequency')
axes[1].axhline(0.5, color='gray', ls=':', lw=1)

print(f'Single-order customers: {(cust_orders["n_orders"] == 1).sum():,} ({(cust_orders["n_orders"] == 1).mean():.1%})')
print(f'Median orders: {cust_orders["n_orders"].median():.0f}')
print(f'Mean orders: {cust_orders["n_orders"].mean():.1f}')
print(f'Max orders: {cust_orders["n_orders"].max()}')

fig.tight_layout()
plt.show()

## 5. Revenue Distribution (Customer Lifetime Value)

In [ ]:
cust_revenue = purchases.groupby('Customer ID')['LineAmount'].sum().reset_index()
cust_revenue.columns = ['Customer ID', 'total_spend']

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(np.log10(cust_revenue['total_spend'].clip(lower=1)), bins=60, color='#2f6db5', edgecolor='white')
ax.set(xlabel='Total spend (log₁₀ £)', ylabel='Customers',
       title='Customer Lifetime Revenue Distribution (log scale)')

# Add percentile annotations
for pct in [50, 90, 99]:
    val = cust_revenue['total_spend'].quantile(pct/100)
    ax.axvline(np.log10(val), color='red', ls='--', alpha=0.7)
    ax.text(np.log10(val), ax.get_ylim()[1]*0.9, f'P{pct}: £{val:,.0f}', rotation=90,
            va='top', ha='right', fontsize=9)

print(f'Median spend: £{cust_revenue["total_spend"].median():,.2f}')
print(f'Mean spend: £{cust_revenue["total_spend"].mean():,.2f}')
print(f'Top 10% of customers account for {cust_revenue.nlargest(int(len(cust_revenue)*0.1), "total_spend")["total_spend"].sum() / cust_revenue["total_spend"].sum():.0%} of revenue')

fig.tight_layout()
plt.show()

## 6. Country Distribution

In [ ]:
country_stats = purchases.groupby('Country').agg(
    customers=('Customer ID', 'nunique'),
    revenue=('LineAmount', 'sum')
).sort_values('revenue', ascending=False)

print(f'UK customers: {country_stats.loc["United Kingdom", "customers"]:,} '
      f'({country_stats.loc["United Kingdom", "customers"]/country_stats["customers"].sum():.0%})')
print(f'UK revenue: £{country_stats.loc["United Kingdom", "revenue"]:,.0f} '
      f'({country_stats.loc["United Kingdom", "revenue"]/country_stats["revenue"].sum():.0%})')

# Top 10 non-UK countries
top_non_uk = country_stats.drop('United Kingdom', errors='ignore').head(10)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(top_non_uk.index[::-1], top_non_uk['revenue'][::-1], color='#2f6db5')
ax.set(xlabel='Revenue (£)', title='Top 10 Non-UK Countries by Revenue')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'£{x:,.0f}'))
fig.tight_layout()
plt.show()

## 7. Return Rate Analysis

In [ ]:
cust_returns = clean.groupby('Customer ID').agg(
    total_invoices=('Invoice', 'nunique'),
    return_invoices=('is_return', 'sum')
).reset_index()
cust_returns['return_rate'] = cust_returns['return_invoices'] / cust_returns['total_invoices']

print(f'Customers with at least one return: {(cust_returns["return_invoices"] > 0).sum():,} '
      f'({(cust_returns["return_invoices"] > 0).mean():.1%})')
print(f'Mean return rate: {cust_returns["return_rate"].mean():.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cust_returns[cust_returns['return_rate'] > 0]['return_rate'], bins=30,
        color='#d4692a', edgecolor='white')
ax.set(xlabel='Return rate (returns / total invoices)', ylabel='Customers',
       title='Return Rate Distribution (customers with ≥1 return)')
fig.tight_layout()
plt.show()

## 8. Recency Distribution (Days Since Last Purchase)

Using Sept 9, 2011 as the snapshot date (matches the test set in the pipeline).

In [ ]:
snapshot = pd.Timestamp('2011-09-09')
last_purchase = purchases.groupby('Customer ID')['InvoiceDate'].max().reset_index()
last_purchase['recency_days'] = (snapshot - last_purchase['InvoiceDate']).dt.days
active = last_purchase[last_purchase['recency_days'] <= 180]  # 180-day active window

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(active['recency_days'], bins=60, color='#2f6db5', edgecolor='white')
ax.axvline(90, color='red', ls='--', lw=2, label='90-day churn threshold')
ax.set(xlabel='Days since last purchase', ylabel='Customers',
       title=f'Recency Distribution (active customers as of {snapshot.date()})')
ax.legend()

print(f'Active customers (last 180 days): {len(active):,}')
print(f'Customers who would be "churned" (recency > 90): {(active["recency_days"] > 90).sum():,} '
      f'({(active["recency_days"] > 90).mean():.1%})')

fig.tight_layout()
plt.show()

## 9. Key Takeaways for Modelling

1. **Data quality**: ~22% of rows have no customer ID; two Excel sheets overlap in Dec 2010 (34K duplicates). Both are handled in the cleaning step.

2. **Heavy-tailed distributions**: Both purchase frequency and revenue are strongly right-skewed → log-transforming monetary features is essential.

3. **Many single-order customers**: A significant fraction of customers placed only one order — the `is_single_order` feature captures this.

4. **Predominantly UK business**: ~90% of customers are UK-based, mostly wholesalers. Results describe this population, not retail consumers in general.

5. **Return behaviour varies**: Most customers have zero returns, but those who return may have useful behavioural signals.

6. **Clear recency signal**: The 90-day threshold provides a natural separation — customers who haven't bought in 90+ days are demonstrably at higher lapse risk.